# CutTrack

Ett verktyg för att logga vikt, kalorier, protein, steg och träning under en deff, och få regelbaserade råd om vikten och kosten rör sig åt rätt håll.

Notebooken byggs upp del för del. I det här steget finns:

- `DailyLog`, klassen som representerar en enskild dags logg
- `User`, basklassen med användarens grunddata och metoderna som räknar på loggarna
- `log_today`, funktionen som frågar användaren om dagens värden och lägger till en logg

`CutProfile`, barnklassen som ärver från `User` och lägger till mål och kaloriberäkningar, byggs i nästa steg.

In [1]:
from datetime import datetime

## DailyLog

Representerar en dags logg. Kontrollerar i `__init__` att vikt och kalorier är rimliga tal. Är de inte det kastas ett `ValueError` med ett förklarande meddelande, som fångas längre ner i `log_today`.

Gränserna (0 till 300 kg, 0 till 10 000 kcal) är satta för att fånga uppenbara skrivfel, som ett extra 0 eller ett minustecken, inte för att vara medicinskt exakta.

`trained` är en bool, `True` om personen tränade den dagen. Under en deff är det träningsfrekvensen som håller muskelmassan uppe, inte hur många minuter passet varade. Ett långt pass med mycket vila ger inte mer stimulans än ett kort och fokuserat. Frekvens går dessutom att bedöma mot en tydlig regel, antal dagar per vecka, medan minuter kräver ett godtyckligt tröskelvärde.

`waist` har standardvärdet `None`, eftersom midjemått är en valfri mätning. Att `None` inte är samma sak som 0 spelar roll längre fram, när snitt ska räknas och skilja på utebliven mätning och ett mätvärde på 0 cm.

In [2]:
class DailyLog:
    """En daglig logg med vikt, kalorier och annan data för ett datum."""

    def __init__(self, date, weight, calories, protein, steps, trained, waist=None):
        if weight <= 0 or weight > 300:
            raise ValueError("Vikten måste vara ett rimligt tal i kilogram, till exempel 82.5.")
        if calories < 0 or calories > 10000:
            raise ValueError("Kalorierna måste vara ett rimligt tal, till exempel 2200.")

        self.date = date
        self.weight = weight
        self.calories = calories
        self.protein = protein
        self.steps = steps
        self.trained = trained
        self.waist = waist

## User

Basklassen för en användare. Innehåller personens grunddata och listan med loggar, plus metoderna som räknar på loggarna.

`logs` börjar som en tom lista. Varje logg som läggs till är ett `DailyLog`-objekt.

Metoderna:

- `add_log`, lägger till en logg, eller byter ut den befintliga om det redan finns en logg för samma datum
- `get_logs(days)`, plockar ut loggarna från de senaste `days` kalenderdagarna
- `average_weight(days)`, medelvikt över perioden
- `weight_change(days)`, hur mycket vikten ändrats över perioden
- `training_days(days)`, antal dagar med träning under perioden

In [3]:
class User:
    """Basklass för en användare av CutTrack."""

    def __init__(self, name, height_cm, age, sex, activity_level, start_weight, created_date):
        self.name = name
        self.height_cm = height_cm
        self.age = age
        self.sex = sex
        self.activity_level = activity_level
        self.start_weight = start_weight
        self.created_date = created_date
        self.logs = []

    def add_log(self, new_log):
        for i in range(len(self.logs)):
            if self.logs[i].date == new_log.date:
                self.logs[i] = new_log
                print(f"Loggen för {new_log.date} uppdaterades.")
                return
        self.logs.append(new_log)
        print(f"Loggen för {new_log.date} lades till.")

    def get_logs(self, days):
        """Returnerar loggarna från de senaste days kalenderdagarna."""
        if len(self.logs) == 0:
            return []

        latest_date = None
        for log in self.logs:
            log_date = datetime.strptime(log.date, "%Y-%m-%d")
            if latest_date is None or log_date > latest_date:
                latest_date = log_date

        selected_logs = []
        for log in self.logs:
            log_date = datetime.strptime(log.date, "%Y-%m-%d")
            days_ago = (latest_date - log_date).days
            if days_ago < days:
                selected_logs.append(log)

        return selected_logs

    def average_weight(self, days):
        """Medelvikt över perioden. Returnerar None om det inte finns några loggar."""
        selected_logs = self.get_logs(days)

        if len(selected_logs) == 0:
            return None

        total_weight = 0
        for log in selected_logs:
            total_weight = total_weight + log.weight

        return total_weight / len(selected_logs)

    def weight_change(self, days):
        """Viktförändring i kg över perioden. Negativt tal betyder att vikten gått ner.
        Returnerar None om det finns färre än två loggar."""
        selected_logs = self.get_logs(days)

        if len(selected_logs) < 2:
            return None

        first_log = selected_logs[0]
        last_log = selected_logs[0]
        for log in selected_logs:
            if log.date < first_log.date:
                first_log = log
            if log.date > last_log.date:
                last_log = log

        return last_log.weight - first_log.weight

    def training_days(self, days):
        """Antal dagar med träning under perioden."""
        selected_logs = self.get_logs(days)

        total_days = 0
        for log in selected_logs:
            if log.trained:
                total_days = total_days + 1

        return total_days

### Hur get_logs räknar kalenderdagar

Enligt teknisk_plan.md ska fönstret räknas på kalenderdagar, inte på antal loggar. Sju loggar utspridda över en månad är inte ett veckosnitt.

Metoden gör tre saker:

1. Loopar igenom alla loggar och hittar det senaste datumet. `datetime.strptime` gör om datumsträngen till ett datumobjekt som går att jämföra med `>`.
2. Loopar igenom alla loggar igen och räknar ut hur många dagar bakåt varje logg ligger, genom att dra det ena datumet från det andra och läsa `.days`.
3. Tar med loggen om den ligger inom fönstret.

`average_weight`, `weight_change` och `training_days` anropar alla `get_logs` först, så urvalsregeln finns på ett enda ställe. Ändras regeln behöver bara `get_logs` ändras.

`weight_change` sorterar inte listan. Den loopar igenom och hittar den tidigaste och den senaste loggen genom att jämföra datumsträngarna direkt. Det fungerar eftersom formatet ÅÅÅÅ-MM-DD sorteras rätt som text, vilket är skälet till att just det formatet valdes i teknisk_plan.md.

`average_weight` och `weight_change` returnerar `None` när underlaget är för tunt, inga loggar alls respektive färre än två. Det skiljer sig från att returnera 0, som skulle betyda att vikten inte ändrades. `training_days` returnerar däremot 0 när det inte finns några loggar, eftersom noll träningsdagar är ett korrekt svar och inte ett saknat värde.

## CutProfile, barnklass med arv

`CutProfile` ärver från `User`. Allt som `User` kan, kan `CutProfile` också: `add_log`, `get_logs`, `average_weight`, `weight_change` och `training_days` finns utan att skrivas om.

Det som läggs till är målen och kaloriberäkningarna, alltså det som är specifikt för någon som deffar. En framtida `BulkProfile` eller `MaintenanceProfile` skulle kunna ärva från samma `User` utan att röra den koden.

`super().__init__(...)` anropar förälderklassens `__init__`, som sätter namn, längd, ålder, kön, aktivitetsnivå, startvikt, datum och den tomma logglistan. Sedan sätter `CutProfile` sina egna attribut. Utan `super()` skulle inget av `User`-attributen finnas, och `self.logs` skulle inte existera.

`protein_goal_per_kg` och `step_goal` har standardvärden, så de kan utelämnas när en profil skapas. `calorie_goal` sätts till `None` från start eftersom den räknas ut först senare, av `suggest_calorie_goal`.

In [4]:
class CutProfile(User):
    """En användare som deffar. Ärver allt från User och lägger till mål och kaloriberäkningar."""

    def __init__(self, name, height_cm, age, sex, activity_level, start_weight, created_date,
                 goal_weight, target_rate_percent, protein_goal_per_kg=1.9, step_goal=8000):
        super().__init__(name, height_cm, age, sex, activity_level, start_weight, created_date)

        if goal_weight <= 0 or goal_weight >= start_weight:
            raise ValueError("Målvikten måste vara lägre än startvikten.")
        if target_rate_percent <= 0 or target_rate_percent > 1.0:
            raise ValueError("Takten måste ligga mellan 0 och 1,0 procent av kroppsvikten per vecka.")

        self.goal_weight = goal_weight
        self.target_rate_percent = target_rate_percent
        self.protein_goal_per_kg = protein_goal_per_kg
        self.step_goal = step_goal
        self.calorie_goal = None

    def current_weight(self):
        """Senaste loggade vikten. Startvikten om inga loggar finns."""
        if len(self.logs) == 0:
            return self.start_weight

        latest_log = self.logs[0]
        for log in self.logs:
            if log.date > latest_log.date:
                latest_log = log

        return latest_log.weight

    def calculate_bmr(self):
        """Basalomsättning i kalorier per dygn, enligt Mifflin-St Jeor."""
        weight = self.current_weight()
        bmr = 10 * weight + 6.25 * self.height_cm - 5 * self.age

        if self.sex == "man":
            bmr = bmr + 5
        else:
            bmr = bmr - 161

        return bmr

    def calculate_tdee(self):
        """Totalt dagligt energibehov, basalomsättningen gånger aktivitetsfaktorn."""
        return self.calculate_bmr() * self.activity_level

    def protein_goal(self):
        """Proteinmål i gram per dag, räknat på målvikten."""
        return self.goal_weight * self.protein_goal_per_kg

### Om beräkningarna

**Mifflin-St Jeor** räknar ut basalomsättningen, alltså hur många kalorier kroppen gör av med i vila. Formeln är densamma för båda könen så när som på sista termen, plus 5 för män och minus 161 för kvinnor. Därför står den gemensamma delen först och if-satsen justerar bara slutet.

**`current_weight`** finns för att BMR ska räknas på vad personen väger nu, inte på startvikten. Under en deff sjunker vikten, och därmed sjunker också förbrukningen. Räknas BMR på startvikten överskattas behovet mer och mer ju längre deffen pågår.

Den hittar senaste loggen genom att jämföra datumsträngar, precis som `weight_change`, och faller tillbaka på `start_weight` när det inte finns några loggar än.

**`calculate_tdee`** multiplicerar med aktivitetsfaktorn: 1,2 stillasittande, 1,375 lätt aktiv, 1,55 måttligt aktiv, 1,725 mycket aktiv, 1,9 extremt aktiv.

**`protein_goal`** räknas på målvikten och inte på nuvarande vikt. Målet ligger därmed fast under hela deffen i stället för att sjunka i takt med att vikten går ner, vilket vore fel: proteinbehovet finns för att skydda den muskelmassa som ska vara kvar när deffen är slut.

**Valideringen** i `__init__` fångar två saker som gör resten av beräkningarna meningslösa: en målvikt som är högre än startvikten, och en takt över 1,0 procent per vecka, där risken för muskelförlust blir för stor. Samma mönster som i `DailyLog`, felet kastas där objektet skapas.

### Test av CutProfile

Testcellen visar tre saker: att formeln stämmer mot handräknat facit, att de ärvda metoderna från `User` fungerar utan att vara omskrivna, och att `isinstance` bekräftar arvet.

In [5]:
profile = CutProfile("Niklas", 183, 39, "man", 1.55, 90.0, "2026-09-01",
                     goal_weight=82.0, target_rate_percent=0.7)

print("Basalomsättning:", round(profile.calculate_bmr(), 1))
print("Energibehov:", round(profile.calculate_tdee(), 1))
print("Proteinmål:", round(profile.protein_goal(), 1), "gram")
print("Aktuell vikt utan loggar:", profile.current_weight())

profile.add_log(DailyLog("2026-09-15", 89.0, 2200, 150, 8000, True))
profile.add_log(DailyLog("2026-09-16", 88.6, 2150, 160, 9000, False))

print("Aktuell vikt med loggar:", profile.current_weight())
print("Basalomsättning nu:", round(profile.calculate_bmr(), 1))
print("Medelvikt 7 dagar:", profile.average_weight(7))
print("Träningsdagar 7 dagar:", profile.training_days(7))
print("Är CutProfile också en User?", isinstance(profile, User))

Basalomsättning: 1853.8
Energibehov: 2873.3
Proteinmål: 155.8 gram
Aktuell vikt utan loggar: 90.0
Loggen för 2026-09-15 lades till.
Loggen för 2026-09-16 lades till.
Aktuell vikt med loggar: 88.6
Basalomsättning nu: 1839.8
Medelvikt 7 dagar: 88.8
Träningsdagar 7 dagar: 1
Är CutProfile också en User? True


In [6]:
# Valideringen ska stoppa orimliga profiler
try:
    CutProfile("Test", 180, 30, "man", 1.2, 80.0, "2026-09-01",
               goal_weight=85.0, target_rate_percent=0.5)
except ValueError as error:
    print("Fångat fel:", error)

try:
    CutProfile("Test", 180, 30, "man", 1.2, 80.0, "2026-09-01",
               goal_weight=75.0, target_rate_percent=1.5)
except ValueError as error:
    print("Fångat fel:", error)

Fångat fel: Målvikten måste vara lägre än startvikten.
Fångat fel: Takten måste ligga mellan 0 och 1,0 procent av kroppsvikten per vecka.


## log_today

Frågar efter dagens värden med `input()`, skapar ett `DailyLog`-objekt och lägger till det på användaren via `add_log`.

Datumformatet kontrolleras med `datetime.strptime`, som kastar `ValueError` om texten inte följer formatet ÅÅÅÅ-MM-DD. Det felet, `ValueError` från `DailyLog` om vikt eller kalorier är orimliga, och `ValueError` från `float()` eller `int()` om någon skriver in text där ett tal förväntas, fångas alla av samma `try/except`, eftersom de är samma feltyp.

Träningsfrågan besvaras med j eller n. Svaret görs om till små bokstäver med `.lower()` så att både J och j fungerar, och jämförs sedan mot strängen `"j"`. Resultatet av jämförelsen är redan `True` eller `False`, så det kan sparas direkt i `trained` utan en if-sats.

Går något fel sparas ingen logg, och felmeddelandet skrivs ut. Användaren får då köra `log_today(user)` igen.

In [ ]:
def log_today(user):
    """Frågar användaren om dagens värden och lägger till en DailyLog på user."""
    date_text = input("Datum (ÅÅÅÅ-MM-DD): ")

    try:
        datetime.strptime(date_text, "%Y-%m-%d")
        weight = float(input("Vikt i kg: "))
        calories = float(input("Kalorier: "))
        protein = float(input("Protein i gram: "))
        steps = int(input("Steg: "))

        trained_text = input("Tränade du idag? (j/n): ")
        trained = trained_text.lower() == "j"

        waist_text = input("Midjemått i cm (lämna tomt om du inte mätt): ")
        if waist_text == "":
            waist = None
        else:
            waist = float(waist_text)

        new_log = DailyLog(date_text, weight, calories, protein, steps, trained, waist)
        user.add_log(new_log)

    except ValueError as error:
        print(f"Loggen sparades inte. Något var fel i inmatningen: {error}")

## Test

Cellen nedan skapar en användare och lägger till fem loggar direkt i koden, så att metoderna går att testa utan att mata in allt för hand.

Den sista loggen ligger långt bak i tiden och ska därför inte räknas med i sjudagarsfönstret, men väl i sextiodagarsfönstret.

In [ ]:
test_user = User("Niklas", 183, 39, "man", 1.55, 90.0, "2026-09-01")

test_user.add_log(DailyLog("2026-09-15", 90.0, 2200, 150, 8000, True))
test_user.add_log(DailyLog("2026-09-16", 89.6, 2150, 160, 9000, False))
test_user.add_log(DailyLog("2026-09-17", 89.8, 2300, 145, 7000, True))
test_user.add_log(DailyLog("2026-09-18", 89.2, 2100, 155, 8500, True))
test_user.add_log(DailyLog("2026-08-01", 95.0, 2500, 120, 5000, False))

print("Loggar totalt:", len(test_user.logs))
print("Loggar i sjudagarsfönstret:", len(test_user.get_logs(7)))
print("Medelvikt 7 dagar:", test_user.average_weight(7))
print("Medelvikt 60 dagar:", test_user.average_weight(60))
print("Viktförändring 7 dagar:", test_user.weight_change(7))
print("Träningsdagar 7 dagar:", test_user.training_days(7))
print("Träningsdagar 60 dagar:", test_user.training_days(60))

### Test med egen inmatning

Kör cellen nedan och fyll i värden själv. Testa både ett giltigt försök och minst ett medvetet felaktigt (fel datumformat, orimlig vikt, text där ett tal förväntas), så att du sett `except`-grenen köras på riktigt.

Kör den sedan en gång till med samma datum som förra gången, för att se att `add_log` uppdaterar posten i stället för att lägga till en till.

In [ ]:
log_today(test_user)

In [ ]:
for log in test_user.logs:
    print(vars(log))